In [30]:
import pandas as pd

# 假设六份DEG结果文件的路径
files = [
    "D:/111/MyDataset_DEG_EG.csv",
    "D:/111/MyDataset_DEG_MG.csv",
    "D:/111/MyDataset_DEG_SG.csv"
]

# --- 第1步：获取所有唯一的细胞类型（group） ---
try:
    df_temp = pd.read_csv(files[0]) 
    celltypes = df_temp['group'].unique().tolist()
    print(f"检测到的细胞类型 (Groups): {celltypes}")
except Exception as e:
    print(f"读取文件 {files[0]} 出错: {e}")
    celltypes = []

# --- 第2步：为每个细胞类型计算交集 ---

# 初始化一个字典来存储每个细胞类型(group)的最终交集结果
all_groups_common_genes = {}

# 外层循环：遍历每一种细胞类型
for group_name in celltypes:
    print(f"\n正在处理细胞类型 (Group): {group_name} ...")
    
    gene_sets_for_this_group = []

    # 内层循环：遍历每一个文件
    for file_path in files:
        try:
            df = pd.read_csv(file_path)
            
            # 筛选条件：p值、logFC 和 *当前正在处理的* group_name
            filtered = df[
                (df['pvals'] < 1e-20) & 
                (df['logfoldchanges'] > 2) & 
                (df['group'] == group_name)
            ]
            
            genes_in_this_file = set(filtered['names'].tolist())
            print(f"  在文件 {file_path.split('/')[-1]} 中找到 {len(genes_in_this_file)} 个符合条件的基因。")
            gene_sets_for_this_group.append(genes_in_this_file)
            
        except Exception as e:
            print(f"  处理文件 {file_path} 时出错: {e}")

    # --- 第3步：计算当前细胞类型在所有文件中的交集 ---
    if gene_sets_for_this_group:
        common_genes = set.intersection(*gene_sets_for_this_group)
        all_groups_common_genes[group_name] = common_genes
    else:
        print(f"  未能为 {group_name} 收集到任何基因集。")
        all_groups_common_genes[group_name] = set()

# --- 第4步：输出所有结果（可选，但有助于调试） ---
print("\n" + "="*30)
print("      所有细胞类型的交集基因结果")
print("="*30)

for group_name, common_genes in all_groups_common_genes.items():
    print(f"\n细胞类型 (Group) '{group_name}':")
    print(f"  在 {len(files)} 个文件中的共同基因数量: {len(common_genes)}")
    if len(common_genes) > 0:
        print(f"  基因列表: {common_genes}")
    else:
        print("  未找到共同基因。")
    print("-" * 20)

# --- 第5步：将结果保存到CSV文件 ---
print("\n正在将结果转换为CSV格式...")

# 创建一个列表来存储所有行
csv_data = []

# 遍历字典，将其转换为 "group", "gene" 的长格式
for group_name, common_genes_set in all_groups_common_genes.items():
    if not common_genes_set:
        # 如果这个group没有交集基因，我们也添加一条记录（基因列为空）
        # 或者选择跳过（取决于你的偏好）
        # csv_data.append({'group': group_name, 'gene': None})
        pass # 这里选择跳过没有交集的
    else:
        # 遍历该group的每一个交集基因
        for gene in common_genes_set:
            csv_data.append({'group': group_name, 'gene': gene})

# 检查是否收集到了数据
if csv_data:
    # 将列表转换为 DataFrame
    results_df = pd.DataFrame(csv_data)
    
    # 定义输出文件名
    output_filename = "all_groups_common_genes.csv"
    
    # 保存为CSV，不保留索引
    try:
        results_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
        print(f"\n🎉 成功！结果已保存到: {output_filename}")
        print(f"总共保存了 {len(results_df)} 条基因记录。")
    except Exception as e:
        print(f"\n保存文件 {output_filename} 时出错: {e}")
        
else:
    print("\n没有找到任何交集基因，未生成CSV文件。")

检测到的细胞类型 (Groups): ['B lymphocytes', 'Basal epithelial cells', 'Endothelial cells', 'Fibroblasts', 'Innate immune cells', 'Keratinized epithelial cells', 'Luminal epithelial cells', 'Proliferating epithelial cells', 'T lymphocytes', 'Vascular smooth muscle cells']

正在处理细胞类型 (Group): B lymphocytes ...
  在文件 MyDataset_DEG_EG.csv 中找到 2356 个符合条件的基因。
  在文件 MyDataset_DEG_MG.csv 中找到 352 个符合条件的基因。
  在文件 MyDataset_DEG_SG.csv 中找到 212 个符合条件的基因。

正在处理细胞类型 (Group): Basal epithelial cells ...
  在文件 MyDataset_DEG_EG.csv 中找到 113 个符合条件的基因。
  在文件 MyDataset_DEG_MG.csv 中找到 255 个符合条件的基因。
  在文件 MyDataset_DEG_SG.csv 中找到 203 个符合条件的基因。

正在处理细胞类型 (Group): Endothelial cells ...
  在文件 MyDataset_DEG_EG.csv 中找到 562 个符合条件的基因。
  在文件 MyDataset_DEG_MG.csv 中找到 690 个符合条件的基因。
  在文件 MyDataset_DEG_SG.csv 中找到 491 个符合条件的基因。

正在处理细胞类型 (Group): Fibroblasts ...
  在文件 MyDataset_DEG_EG.csv 中找到 731 个符合条件的基因。
  在文件 MyDataset_DEG_MG.csv 中找到 509 个符合条件的基因。
  在文件 MyDataset_DEG_SG.csv 中找到 385 个符合条件的基因。

正在处理细胞类型 (Group): Innate immune ce

In [32]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import pandas as pd

# ---------- 1. 加载数据并构建 gene_dict ----------
# (这是修改的部分，承接上一个脚本的输出)

input_csv = "all_groups_common_genes.csv" # 这是上一步保存的文件名

try:
    print(f"正在从 {input_csv} 加载数据...")
    df_input = pd.read_csv(input_csv)
    
    # 将长格式的 DataFrame 转换为字典
    # 键(key)是 'group'，值(value)是该 group 对应的所有基因的 *集合 (set)*
    gene_dict = df_input.groupby('group')['gene'].apply(set).to_dict()
    
    print(f"成功加载了 {len(gene_dict)} 个 group 的数据。")
    # print(gene_dict) # 取消注释以查看加载的字典

except FileNotFoundError:
    print(f"错误：未找到文件 '{input_csv}'。")
    print("请确保上一个脚本已成功运行，并生成了此文件。")
    gene_dict = {} # 给一个空字典以防止后续代码崩溃
except Exception as e:
    print(f"读取 {input_csv} 时出错: {e}")
    gene_dict = {}


# ---------- 2. (可选) 合并 MaSC 与 MaSC-t2-sg ----------
# (这部分逻辑已变得更健壮，只在组存在时才合并)

if 'MaSC' in gene_dict and 'MaSC-t2-sg' in gene_dict:
    print("检测到 MaSC 和 MaSC-t2-sg，正在合并...")
    # 因为 gene_dict 的值现在是 set，我们可以直接用 | (并集) 操作
    combined_masc = gene_dict['MaSC'] | gene_dict['MaSC-t2-sg']
    gene_dict['MaSC'] = combined_masc     # 覆盖原 MaSC
    gene_dict.pop('MaSC-t2-sg')         # 删除旧键
    print("合并完成。")
else:
    print("未检测到 MaSC 和 MaSC-t2-sg，跳过合并步骤。")


# ---------- 3. 计算各组唯一基因 ----------
# (这部分逻辑与你提供的完全相同，但进行了一点优化)

print("正在计算各组的独有基因...")
unique_dict = {}
all_group_keys = list(gene_dict.keys()) # 获取一次所有键

for key in all_group_keys:
    # 创建一个包含 *所有其他* 组基因的并集
    genes_in_other_groups = set().union(
        *[gene_dict[k] for k in all_group_keys if k != key]
    )
    
    # 使用集合的 "差集" 运算 ( - )
    # 找出在当前组 (gene_dict[key]) 中，但不在 "其他组并集" 中的基因
    unique_genes = gene_dict[key] - genes_in_other_groups
    
    # 将结果（一个集合）转换为排序后的列表
    unique_dict[key] = sorted(list(unique_genes))


# ---------- 4. 转成长格式 DataFrame ----------
# (这部分逻辑与你提供的完全相同)

print("正在将结果转换为长格式 DataFrame...")
unique_df = (
    pd.Series(unique_dict, name="gene_list")
      .explode()                   # 每行一个基因
      .reset_index()               # columns=['group', 'gene']
      .rename(columns={"index": "group", "gene_list": "gene"})
      .dropna()                    # 移除没有任何独有基因的组 (产生的 NaN)
)


# ---------- 5. 保存结果 ----------
# (这部分逻辑与你提供的完全相同)
output_filename = "unique_common_genes_long.csv"
try:
    unique_df.to_csv(output_filename, index=False)
    print(f"\n🎉 成功！独有基因列表已保存到: {output_filename}")
except Exception as e:
    print(f"保存文件 {output_filename} 时出错: {e}")


# ---------- 6. 打印总结 ----------
if __name__ == "__main__":
    print("\n▶ 独有基因数量 (来自'all_groups_common_genes.csv')：")
    total_unique_genes = 0
    for k, v in unique_dict.items():
        count = len(v)
        total_unique_genes += count
        print(f"  {k:15s} : {count:4d} 个")
    
    print(f"  --------------------------")
    print(f"  {'Total':15s} : {total_unique_genes:4d} 个")


    if not unique_df.empty:
        print("\n▶ DataFrame 示例：")
        print(unique_df.head(10))
    else:
        print("\n▶ DataFrame 示例：\n  (未找到任何独有基因)")

正在从 all_groups_common_genes.csv 加载数据...
成功加载了 10 个 group 的数据。
未检测到 MaSC 和 MaSC-t2-sg，跳过合并步骤。
正在计算各组的独有基因...
正在将结果转换为长格式 DataFrame...

🎉 成功！独有基因列表已保存到: unique_common_genes_long.csv

▶ 独有基因数量 (来自'all_groups_common_genes.csv')：
  B lymphocytes   :   11 个
  Basal epithelial cells :   28 个
  Endothelial cells :  222 个
  Fibroblasts     :  218 个
  Innate immune cells :  215 个
  Keratinized epithelial cells :   11 个
  Luminal epithelial cells :   25 个
  Proliferating epithelial cells :  515 个
  T lymphocytes   :   58 个
  Vascular smooth muscle cells :   50 个
  --------------------------
  Total           : 1353 个

▶ DataFrame 示例：
           group     gene
0  B lymphocytes   Atp8a1
1  B lymphocytes     Blnk
2  B lymphocytes    Bmp2k
3  B lymphocytes    Edem1
4  B lymphocytes    Fnbp1
5  B lymphocytes  Herpud1
6  B lymphocytes  Ralgps2
7  B lymphocytes   Sdf2l1
8  B lymphocytes    Stt3b
9  B lymphocytes      Tec


In [9]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Compute cross-gland mean expression for a *celltype-aware* conserved-gene list,
then build a unique Top-N `selection` dict (one gene belongs to one celltype only).

Inputs
------
1. unique_common_genes_long.csv  ← long-format, columns: celltype,gene,[...]
2. single_adata_normalized.h5ad  ← ONE file, with .X as normalized data
   └── AnnData.obs[GROUP_KEY]     ← cell-type annotation (e.g., 'newcelltype')
   └── AnnData.obs[GLAND_KEY]     ← gland annotation (e.g., 'gland')

Outputs
-------
1. cross_gland_gene_expression.csv ← long table: celltype,gene,cross_gland_mean,EG_mean,MG_mean,SG_mean
2. Prints + returns `selection`      ← {celltype: [Top-N genes]}
"""

from pathlib import Path
from collections import OrderedDict
import numpy as np
import pandas as pd
import scanpy as sc

# --------------------------- CONFIG ---------------------------
### (已修改)
DATA_ROOT   = Path(r"D:/111")
# ↓↓↓ 你的单个 .h5ad 文件名
H5AD_FILE   = "YOUR_SINGLE_ANNDATA.h5ad" 
# ↓↓↓ 你的基因列表 (来自上一步)
GENE_CSV    = Path("unique_common_genes_long.csv") 

# ↓↓↓ 预处理后你关心的腺体 (必须与 rename_and_merge_glands 的输出匹配)
GLANDS_TO_ANALYZE = ["EG", "MG", "SG"] 

# ↓↓↓ AnnData.obs 中用于细胞类型和腺体的列名
GROUP_KEY   = "newcelltype"   # 细胞类型 (e.g., MaSC, Luminal)
GLAND_KEY   = "gland"         # 腺体 (e.g., MG, SG, AG, CG)

TOP_N       = 5
CSV_OUT     = "cross_gland_gene_expression.csv" # (已重命名)
# ---------------------------------------------------------------


# ---------- 1. (新增) 你的预处理函数 ----------
def rename_and_merge_glands(adata):
    """Apply:
     1) SG -> EG
     2) AG -> SG, CG -> SG  (after step 1)
    """
    if GLAND_KEY not in adata.obs.columns:
        raise ValueError(f"缺少 adata.obs['{GLAND_KEY}'] 列。")

    print(f"[INFO] 正在重命名/合并 '{GLAND_KEY}' 列...")
    print(f"  原始计数:\n{adata.obs[GLAND_KEY].value_counts().to_string()}")

    # Step 1: SG -> EG
    adata.obs[GLAND_KEY] = adata.obs[GLAND_KEY].replace({'SG': 'EG'})
    # Step 2: merge AG/CG into SG
    adata.obs[GLAND_KEY] = adata.obs[GLAND_KEY].replace({'AG': 'SG', 'CG': 'SG'})

    print("[OK] 已重命名/合并 gland：SG→EG；AG+CG→SG。")
    print(f"[INFO] 重命名后 '{GLAND_KEY}' 计数：")
    print(adata.obs[GLAND_KEY].value_counts().to_string())
    return adata


# ---------- 2. (不变) 读取 celltype↔gene 映射 ----------
def load_mapping(csv_path: Path) -> dict[str, list[str]]:
    """
    读取CSV，构建一个 {celltype -> [genes]} 的字典。
    关键：通过'seen'集合确保每个基因只被分配给它在CSV中*首次*出现的celltype。
    """
    df = pd.read_csv(csv_path)

    df.columns = [str(c).lower() for c in df.columns]

    # 自动检测 'celltype' 和 'gene' 列，否则使用前两列
    col_map = {}
    if "celltype" in df.columns:
        col_map["celltype"] = "celltype"
    else:
        print(f"[WARN] 未找到 'celltype' 列，使用第1列 '{df.columns[0]}'")
        col_map["celltype"] = df.columns[0]
        
    if "gene" in df.columns:
        col_map["gene"] = "gene"
    else:
        print(f"[WARN] 未找到 'gene' 列，使用第2列 '{df.columns[1]}'")
        col_map["gene"] = df.columns[1]

    if df.shape[1] < 2:
        raise ValueError("CSV needs at least two columns (celltype, gene).")
    df = df.rename(columns={v: k for k, v in col_map.items()})


    mapping: dict[str, list[str]] = OrderedDict()
    seen = set()
    for ct, g in zip(df["celltype"], df["gene"]):
        if pd.isna(ct) or pd.isna(g) or g in seen:
            continue
        mapping.setdefault(ct, []).append(g)
        seen.add(g)
    return mapping


# ---------- 3. (已修改) 计算指定细胞类型/腺体中一组基因的均值 ----------
def mean_expr_gland(adata: sc.AnnData, genes: list[str], celltype: str, gland: str) -> pd.Series:
    """
    (替换了 mean_expr_species)
    返回一个物种中，一个 celltype 在一个 gland 中的基因平均表达。
    """
    
    # 检查细胞类型和腺体是否存在于 .obs 中
    if celltype not in adata.obs[GROUP_KEY].values:
        print(f"  [WARN] {celltype} 不在 {GROUP_KEY} 中。")
        return pd.Series(np.nan, index=genes)
    if gland not in adata.obs[GLAND_KEY].values:
        print(f"  [WARN] {gland} 不在 {GLAND_KEY} 中。")
        return pd.Series(np.nan, index=genes)

    # 取目标细胞 (必须同时匹配 celltype 和 gland)
    idx_cells = (adata.obs[GROUP_KEY] == celltype) & (adata.obs[GLAND_KEY] == gland)
    
    if idx_cells.sum() == 0:
        # print(f"  [Debug] {celltype} + {gland} 中没有细胞。")
        return pd.Series(np.nan, index=genes)

    # 限制到存在于该 AnnData 的基因，保持原顺序
    valid_genes = [g for g in genes if g in adata.var_names]
    if not valid_genes:
        print(f"  [WARN] {celltype} 的基因列表在 anndata.var_names 中均未找到。")
        return pd.Series(np.nan, index=genes)

    # (已修改) 从 .X 提取表达矩阵并求均值
    X = adata[idx_cells, valid_genes].X  # 使用 .X
    
    means = np.asarray(X.mean(axis=0)).ravel()
    return pd.Series(means, index=valid_genes).reindex(genes) # 保证长度一致


# ---------- 4. (已修改) 构建长格式表达表 ----------
def build_expression_table(adata: sc.AnnData, mapping: dict) -> pd.DataFrame:
    """
    (替换了 adatas: dict)
    遍历所有 celltype，计算其在每个 gland 中的平均表达，然后计算跨腺体均值。
    """
    recs = []
    print("\n[INFO] 正在计算所有 celltype 和 gland 组合的平均表达...")
    
    for ct, genes in mapping.items():
        # print(f"  正在处理 {ct}...") # (如果需要详细日志，取消注释)
        
        # (已修改) 不再遍历 adatas 字典，而是遍历 GLANDS_TO_ANALYZE 列表
        gland_means = {
            gland: mean_expr_gland(adata, genes, ct, gland)
            for gland in GLANDS_TO_ANALYZE
        }
        
        stacked = pd.DataFrame(gland_means)
        
        # (已修改) 计算 cross_gland_mean
        stacked["cross_gland_mean"] = stacked.mean(axis=1, skipna=True)
        
        # (已修改) 动态写入记录
        recs.extend(
            {
                "celltype": ct,
                "gene": g,
                "cross_gland_mean": row["cross_gland_mean"],
                # 动态添加每个gland的均值
                **{f"{gland}_mean": row.get(gland, np.nan) for gland in GLANDS_TO_ANALYZE}
            }
            for g, row in stacked.iterrows()
        )
    print("[OK] 表达表计算完成。")
    return pd.DataFrame.from_records(recs)


# ---------- 5. (已修改) 生成 Top-N selection ----------
def build_selection(expr_df: pd.DataFrame, mapping: dict) -> dict:
    """
    基于 'cross_gland_mean' 进行排序和筛选。
    """
    selection = {}
    used = set()  # 已入选基因
    print("\n[INFO] 正在构建唯一的 Top-N 基因选择...")
    
    # 保持 mapping 中 celltype 的顺序 (优先级)
    for ct in mapping:
        top_genes = (
            expr_df.query("celltype == @ct and gene not in @used")
                   # (已修改) 按 cross_gland_mean 排序
                   .sort_values("cross_gland_mean", ascending=False)
                   .head(TOP_N)["gene"]
                   .tolist()
        )
        selection[ct] = top_genes
        used.update(top_genes)
        
    print("[OK] Top-N 字典构建完成。")
    return selection


# ---------- 6. (已修改) 主流程 ----------
def main():
    # 1) 读取映射
    mapping_path = DATA_ROOT / GENE_CSV
    mapping = load_mapping(mapping_path)
    print(f"[OK] 基因映射加载完成: {sum(map(len, mapping.values()))} 个唯一基因 "
          f"分布在 {len(mapping)} 个 celltype 中 (来自 {mapping_path.name})")

    # 2) 载入单个 AnnData
    adata_path = DATA_ROOT / H5AD_FILE
    print(f"[INFO] 正在加载 AnnData: {adata_path} ...")
    ad = sc.read_h5ad(adata_path)
    print(f"[OK] AnnData 加载: {ad.n_obs:,} 个细胞, {ad.n_vars:,} 个基因。")
    
    # 验证 .obs 列
    if GROUP_KEY not in ad.obs.columns:
        raise KeyError(f"{adata_path} - 缺少细胞类型列 .obs['{GROUP_KEY}']")
    if GLAND_KEY not in ad.obs.columns:
        raise KeyError(f"{adata_path} - 缺少腺体列 .obs['{GLAND_KEY}']")
        
    print(f"[INFO] 将使用 adata.X 作为归一化表达数据。")

    # 3) (新增) 运行预处理
    ad = rename_and_merge_glands(ad)

    # 4) 表达量统计
    expr_df = build_expression_table(ad, mapping)
    
    output_path = DATA_ROOT / CSV_OUT
    expr_df.to_csv(output_path, index=False)
    print(f"\n[OK] 跨腺体表达CSV已保存 → {output_path} ({len(expr_df):,} 行)")

    # 5) Top-N selection
    selection = build_selection(expr_df, mapping)
    print("\n" + "="*30)
    print(f"    Top-{TOP_N} 'selection' 字典    ")
    print("="*30)
    for ct, genes in selection.items():
        print(f"{ct:<15} : {genes}")

    return selection


if __name__ == "__main__":
    # 确保脚本在DATA_ROOT目录运行，或者GENE_CSV和H5AD_FILE在正确路径
    try:
        selection = main()
    except FileNotFoundError as e:
        print(f"\n[ERROR] 文件未找到: {e}")
        print("请检查 CONFIG 部分中的 DATA_ROOT, H5AD_FILE, 和 GENE_CSV 路径是否正确。")
    except KeyError as e:
        print(f"\n[ERROR] 键错误: {e}")
        print(f"请检查 CONFIG 中的 GROUP_KEY ('{GROUP_KEY}') 或 GLAND_KEY ('{GLAND_KEY}') "
              "是否与 .h5ad 文件中的 .obs 列名匹配。")

[INFO] Mapping loaded: 264 unique genes across 5 celltypes
[INFO] M-MG: 10,880 cells, 12,088 genes.
[INFO] R-MG: 19,235 cells, 12,088 genes.
[INFO] S-MG: 22,855 cells, 12,088 genes.
[INFO] CSV saved → conserved_gene_expression.csv  (264 rows)

=== Top-5 'selection' dict ===
LumSEC-Lac   : ['Lalba', 'Fabp3', 'Mfge8', 'Pigr', 'Epcam']
LumSEC-Lip   : ['Pdk4', 'Ltf', 'Cpeb4', 'Ppm1h', 'Rasef']
LumHR        : ['Trps1', 'Prlr', 'Gata3', 'Esr1', 'Ano1']
Basal        : ['Acta2', 'Tpm2', 'Myh11', 'Krt17', 'Cald1']
MaSC         : ['Mif', 'Fabp5', 'Dcn', 'Gstm5', 'Snrpd1']


In [3]:

#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Batch heat-map of Z-scored gene expression for a SINGLE AnnData file,
split by 'gland', using a celltype→gene dictionary (selection).

Author: <your name>
Date  : 2025-11-05
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1️⃣ 你的 celltype→Top5 基因字典
# (请将你上一步 'build_selection' 脚本输出的 'selection' 字典粘贴到这里)
# ------------------------------------------------------------------
selection = {
    # 示例格式 (请用你自己的真实数据替换):
    'B lymphocytes': ['Krt10', 'Kmo', 'Plaur', 'Nr4a2', 'Crabp2'],
    'Basal epithelial cells': ['Nrp2', 'Mbp', 'Cd44', 'Nfatc2', 'Adamts9'],
    'Endothelial cells': ['Chrm3', 'Gem', 'Fbln2', 'Mdga1', 'Smad3'],
    'Fibroblasts': ['Irf1', 'Tm7sf3', 'Galnt2', 'Tns3', 'Pfn4'],
    'Innate immune cells': ['Itga5', 'Bmp6', 'Cd81', 'Dnajb11', 'F8'],
    'Keratinized epithelial cells': ['Mef2c', 'Ly86', 'Lyn', 'Cd83', 'Cited4'],
    'Luminal epithelial cells': ['Angpt1', 'Smpd3', 'Col17a1', 'Ar', 'Trim29'],
    'Proliferating epithelial cells': ['Aqp2', 'Irx4', 'Frmd5', 'Aadat', 'Nedd4'],
    'T lymphocytes': ['Adam33', 'Bicd1', 'Lrrc28', 'Syt11', 'Spopl'],
    'Vascular smooth muscle cells': ['Serpinh1', 'Tnmd', 'Kcnt2', 'Esrrg', 'Rasip1']
    # ... 在此粘贴你的完整字典 ...
}

# 衍生列表，保持顺序
celltype_list = list(selection.keys())
gene_list     = [g for ct in celltype_list for g in selection[ct]] 

print(f"[INFO] 成功加载 'selection' 字典。")
print(f"[INFO] 总共 {len(celltype_list)} 个细胞类型, {len(gene_list)} 个基因。")


# ------------------------------------------------------------------
# 2️⃣ 其他参数 (已修改)
# ------------------------------------------------------------------
adata_root = Path(r"D:/111")
# ↓↓↓ 你的单个 .h5ad 文件名 (包含所有腺体)
H5AD_FILE  = "YOUR_SINGLE_ANNDATA.h5ad" 

# ↓↓↓ 预处理后要绘图的腺体
GLANDS_TO_PLOT = ["EG", "MG", "SG"] 

# ↓↓↓ AnnData.obs 中用于细胞类型和腺体的列名
group_key  = "anno"          # (已修改) obs 中细胞类型列
gland_key  = "gland"         # obs 中腺体列

# (已恢复) 恢复你原来的配色方案
cmap       = sns.light_palette("#404040", n_colors=256, as_cmap=True) 
dpi        = 300

sns.set(style="white", font_scale=0.8)

# ------------------------------------------------------------------
# 3️⃣ 辅助函数 (用于预处理)
# ------------------------------------------------------------------
def rename_and_merge_glands(adata, gland_col):
    """
    应用: 1) SG -> EG; 2) AG -> SG, CG -> SG
    """
    if gland_col not in adata.obs.columns:
        raise ValueError(f"缺少 adata.obs['{gland_col}'] 列。")
    
    print(f"[INFO] 正在重命名/合并 '{gland_col}' 列...")
    # 确保是字符串，以便替换
    if pd.api.types.is_categorical_dtype(adata.obs[gland_col]):
        adata.obs[gland_col] = adata.obs[gland_col].astype(str)

    # Step 1: SG -> EG
    adata.obs[gland_col] = adata.obs[gland_col].replace({'SG': 'EG'})
    # Step 2: merge AG/CG into SG
    adata.obs[gland_col] = adata.obs[gland_col].replace({'AG': 'SG', 'CG': 'SG'})

    # 转回 Category
    adata.obs[gland_col] = adata.obs[gland_col].astype("category")
    print(f"[OK] 腺体合并完成。")
    print(f"[INFO] 重命名后 '{gland_col}' 计数：\n{adata.obs[gland_col].value_counts().to_string()}")
    return adata

# (MaSC 合并函数已移除)

# ------------------------------------------------------------------
# 4️⃣ 主流程 (已修改)
# ------------------------------------------------------------------

print("\n" + "="*30)
print("     开始执行热图绘制流程")
print("="*30)

# --- 1. 加载和预处理 (在循环外) ---
adata_path = adata_root / H5AD_FILE
if not adata_path.exists():
    raise FileNotFoundError(adata_path)

print(f"[INFO] 正在加载主 AnnData: {adata_path} ...")
adata_main = sc.read_h5ad(adata_path)
print(f"[OK] 主 AnnData 加载: {adata_main.n_obs:,} 个细胞, {adata_main.n_vars:,} 个基因。")

# 运行你的预处理
adata_main = rename_and_merge_glands(adata_main, gland_key)
# (MaSC 合并步骤已移除)

# 假设 .X 已经是归一化数据 (基于你的上一个脚本)
print(f"[INFO] 将使用 adata.X 作为归一化表达数据。")

# 提取颜色映射 (如果存在)
ct_color_map = None
if f"{group_key}_colors" in adata_main.uns:
    print("[INFO] 找到了细胞类型颜色映射。")
    # 确保使用 .obs 中的 categories 来正确匹配颜色
    if pd.api.types.is_categorical_dtype(adata_main.obs[group_key]):
        ct_color_map = dict(zip(
            adata_main.obs[group_key].cat.categories,
            adata_main.uns[f"{group_key}_colors"]
        ))
    else:
        print("[WARN] group_key 不是 category 类型，颜色映射可能不准确。")
        # 尝试基于 unique 值创建，但不保证顺序
        ct_color_map = dict(zip(
            np.unique(adata_main.obs[group_key]),
            adata_main.uns[f"{group_key}_colors"]
        ))


# --- 2. 主循环 (遍历腺体) ---
for gland in GLANDS_TO_PLOT:
    print(f"\n[INFO] Processing Gland: {gland} …")
    
    # (新增) 拆分 (subset)
    adata_gland = adata_main[adata_main.obs[gland_key] == gland].copy()
    
    if adata_gland.n_obs == 0:
        print(f"  [WARN] 在 {gland_key} 中找不到 {gland} 的细胞，跳过。")
        continue

    # ——— 基因 & 细胞类型实际可用列表 ————————————————
    genes_avail     = [g for g in gene_list if g in adata_gland.var_names]
    celltypes_avail = [ct for ct in celltype_list if ct in adata_gland.obs[group_key].unique()]
    
    if not genes_avail or not celltypes_avail:
        print(f"  [WARN] {gland} 中缺少足够的基因或细胞类型，跳过。")
        continue
    
    print(f"  [INFO] 找到 {len(celltypes_avail)} 个细胞类型和 {len(genes_avail)} 个基因。")

    # ——— 构建平均表达矩阵 gene × celltype ————————————
    expr = pd.DataFrame(index=genes_avail, columns=celltypes_avail, dtype=float)

    for ct in celltypes_avail:
        cells_mask = adata_gland.obs[group_key] == ct
        adata_subset = adata_gland[cells_mask, genes_avail]
        mean_vec = np.asarray(adata_subset.X.mean(axis=0)).ravel()
        expr.loc[genes_avail, ct] = mean_vec

    # ——— Z-score (行) & 旋转 ——————————
    expr_mean = expr.mean(axis=1)
    expr_std = expr.std(axis=1).replace(0, 1) # 替换0标准差
    
    expr_z = expr.sub(expr_mean, axis=0)
    expr_z = expr_z.div(expr_std, axis=0)
    
    heat   = expr_z.T  # 旋转: 行=celltype, 列=gene

    # 保证 row/col 顺序完全按 selection (的子集)
    heat = heat.loc[celltypes_avail, genes_avail]

    # ——— 行颜色条 ————————————————
    row_colors = None
    if ct_color_map:
        row_colors = [ct_color_map.get(ct, '#808080') for ct in heat.index] # 使用.get以防万一

    # ——— 画图 ——————————————
    figsize = (max(6, len(genes_avail) * 0.35),
               max(3, len(celltypes_avail) * 0.4))

    print(f"  [INFO] 正在绘制热图 (FigSize: {figsize[0]:.1f} x {figsize[1]:.1f})...")
    g = sns.clustermap(
        heat,
        row_cluster=False, col_cluster=False, 
        cmap=cmap,                          # (恢复) 灰色调
        vmin=0, vmax=2,                     # (恢复) 0-2 范围
        row_colors=row_colors,
        linewidths=0.1,
        figsize=figsize
    )

    g.ax_heatmap.set_xlabel("Gene")
    g.ax_heatmap.set_ylabel("Cell type")
    g.ax_heatmap.set_title(gland, pad=12) # 标题为腺体名
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90)

    out_name = f"{gland}_zscore_heatmap.pdf"
    g.savefig(out_name, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"  [OK] Saved → {out_name}")

print("\n✅ All heatmaps generated!")
```

[INFO] Processing M-MG …
  [OK] Saved → M-MG_zscore_heatmap.pdf
[INFO] Processing R-MG …
  [OK] Saved → R-MG_zscore_heatmap.pdf
[INFO] Processing S-MG …


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25560\3683509129.py:55: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(adata.obs["newcelltype"]):


  [OK] Saved → S-MG_zscore_heatmap.pdf

✅ All heatmaps generated!


In [1]:
#!/usr/bin/env python
# coding: utf-8
"""
Gland-specific DEG 筛选 & Heatmap（自动检测所有细胞类型）
 - 使用 adata.layers['normalized'] 作为表达矩阵
 - 行顺序自动按检测到的细胞类型排序
"""

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# --- (已修改) ---
# 你的单个 AnnData 文件
H5AD_FILE = "YOUR_SINGLE_ANNDATA.h5ad"#log1p

# 要分析的腺体 (在预处理后)
GLANDS = ["EG", "MG", "SG"]

# 对应 DEG csv 路径 (!! 假设你已按腺体准备了这些文件 !!)
DEG_PATH = {
    gland: Path(f"D:/111/{gland}_ranked_genes.csv") 
    for gland in GLANDS
}
# --- (结束修改) ---

# (已移除) 不再需要 CELLTYPE_ORDER

ADATA_ROOT = Path(r"D:/111")
GROUP_KEY  = "anno" # 细胞类型列
GLAND_KEY  = "gland"       # 腺体列 (拆分会用到)
ZERO_EXPR_THRESHOLD = 0.01
TOP_N      = 5
HEAT_CMAP  = "flare_r"
DPI        = 300

# Gland 调色板
GLAND_CMAP = {
    "EG": sns.light_palette("#7BC6E6", n_colors=256, as_cmap=True), # 蓝色
    "MG": sns.light_palette("#E5B764", n_colors=256, as_cmap=True), # 黄色
    "SG": sns.light_palette("#A3C29F", n_colors=256, as_cmap=True), # 绿色
}
# ------------------------------------------------------------------


# 腺体预处理函数
def rename_and_merge_glands(adata, gland_col):
    """
    应用: 1) SG -> EG; 2) AG -> SG, CG -> SG
    """
    if gland_col not in adata.obs.columns:
        raise ValueError(f"缺少 adata.obs['{gland_col}'] 列。")
    
    print(f"[INFO] 正在重命名/合并 '{gland_col}' 列...")
    if pd.api.types.is_categorical_dtype(adata.obs[gland_col]):
        adata.obs[gland_col] = adata.obs[gland_col].astype(str)

    adata.obs[gland_col] = adata.obs[gland_col].replace({'SG': 'EG'})
    adata.obs[gland_col] = adata.obs[gland_col].replace({'AG': 'SG', 'CG': 'SG'})

    adata.obs[gland_col] = adata.obs[gland_col].astype("category")
    print(f"[OK] 腺体合并完成。")
    print(f"[INFO] 重命名后 '{gland_col}' 计数：\n{adata.obs[gland_col].value_counts().to_string()}")
    return adata


# (已修改) 加载、预处理并按 Gland 拆分
def load_and_split_anndata():
    """
    读 1 份 AnnData，预处理，按 Gland 拆分到 dict{gland: AnnData}
    (已移除 MaSC 合并)
    (新增) 返回自动检测到的细胞类型列表
    """
    print(f"[INFO] 正在加载主 AnnData: {H5AD_FILE} ...")
    f = H5AD_FILE
    if not f.exists():
        raise FileNotFoundError(f)
    
    ad_main = sc.read_h5ad(f)
    print(f"[OK] 主 AnnData 加载: {ad_main.n_obs:,} 个细胞, {ad_main.n_vars:,} 个基因。")

    # 1. (已移除) MaSC 合并逻辑
    
    # (新增) 自动检测所有细胞类型，并排序
    if pd.api.types.is_categorical_dtype(ad_main.obs[GROUP_KEY]):
        all_celltypes = ad_main.obs[GROUP_KEY].cat.categories.tolist()
    else:
        all_celltypes = sorted(ad_main.obs[GROUP_KEY].unique().tolist())
    print(f"[INFO] 自动检测到 {len(all_celltypes)} 个细胞类型 (按此排序):")
    print(all_celltypes)

    # 2. 运行腺体预处理
    ad_main = rename_and_merge_glands(ad_main, GLAND_KEY)
    
    # 3. 检查 'normalized' layer
    if 'normalized' not in ad_main.layers:
        raise KeyError(f"{f} 缺少 layers['normalized']")
    print("[INFO] 确认存在 layers['normalized']。")

    # 4. 按 Gland 拆分
    ads_dict = {}
    for gland in GLANDS:
        adata_subset = ad_main[ad_main.obs[GLAND_KEY] == gland].copy()
        if adata_subset.n_obs > 0:
            ads_dict[gland] = adata_subset
            print(f"  > 拆分 '{gland}': {adata_subset.n_obs:,} 个细胞。")
        else:
            print(f"  [WARN] '{gland}' 中没有细胞，将跳过此组。")
            
    # (修改) 返回字典和细胞类型列表
    return ads_dict, all_celltypes


def genes_zero_in_others(genes, other_adatas):
    """(逻辑不变) 过滤：在所有“其他腺体”里 ≤ N % 细胞表达的基因集合."""
    keep = set(genes)
    for ad in other_adatas:
        genes_to_check = [g for g in keep if g in ad.var_names]
        if not genes_to_check:
            continue
            
        mat_other = (ad[:, genes_to_check].layers['normalized'] > 0)
        prop = mat_other.mean(axis=0).A1
        gene_prop_map = dict(zip(genes_to_check, prop))
        
        keep -= {g for g in keep if gene_prop_map.get(g, 0) > ZERO_EXPR_THRESHOLD}
        
    return keep


# (已修改) 
def filter_gland_specific(gland, ads_dict, celltypes_to_keep):
    """
    (替换 filter_species_specific)
    (修改) 使用传入的 celltypes_to_keep 列表进行筛选
    """
    if gland not in ads_dict:
        print(f"  [WARN] {gland} 不在已加载的 anndata 字典中，跳过。")
        return {}
        
    ad_target = ads_dict[gland]
    others    = [ads_dict[g] for g in ads_dict if g != gland]

    zero_in_others = genes_zero_in_others(ad_target.var_names, others)
    print(f"    > {len(zero_in_others)} 个基因在其他腺体中低表达。")

    deg_file = DEG_PATH[gland]
    if not deg_file.exists():
        print(f"  [ERROR] 找不到DEG文件: {deg_file}。跳过 {gland}。")
        return {}
        
    deg = pd.read_csv(deg_file)
    deg = deg[deg["gene"].isin(zero_in_others)]
    deg = deg.sort_values("logfoldchange", ascending=False)

    # (修改) 按自动检测到的细胞类型列表筛选
    out = {ct: df for ct, df in deg.groupby("group") if ct in celltypes_to_keep}
    return out


def save_deg_csv(ct_dict, gland):
    """(逻辑不变)"""
    if not ct_dict:
        return None
    df = pd.concat(
        [d.assign(celltype=ct) for ct, d in ct_dict.items()],
        ignore_index=True
    )
    if df.empty:
        return None
    out = f"{gland}_specific_DEG.csv"
    df.to_csv(out, index=False)
    return out


# (已修改)
def top_genes(ct_dict, celltype_order_list, n=TOP_N):
    """(修改) 使用传入的 celltype_order_list 保证顺序"""
    genes = []
    # (修改) 按自动检测的顺序遍历
    for ct in celltype_order_list: 
        if ct in ct_dict:
            genes += ct_dict[ct].nlargest(n, 'logfoldchange')['gene'].tolist()
    
    seen = set(); ordered = []
    for g in genes:
        if g not in seen:
            ordered.append(g); seen.add(g)
    return ordered


# (已修改)
def plot_heatmap(genes, adata, gland_tag, gene_list_tag, celltype_order_list):
    """
    (修改) 使用传入的 celltype_order_list 保证行顺序
    """
    
    genes_avail = [g for g in genes if g in adata.var_names]
    if not genes_avail:
        print(f"  [WARN] 基因列表 {gene_list_tag} 在 {gland_tag} 数据中均不存在。")
        return None

    # (修改) 按自动检测的顺序筛选
    present_ct = [ct for ct in celltype_order_list if ct in adata.obs[GROUP_KEY].unique()]
    if not present_ct:
        print(f"  [WARN] {gland_tag} 数据中不存在 {celltype_order_list} 中的任何细胞类型。")
        return None

    expr = pd.DataFrame(index=genes_avail, columns=present_ct, dtype=float)
    var_lookup = pd.Series(np.arange(adata.n_vars), index=adata.var_names)

    for ct in present_ct:
        cells = adata.obs_names[adata.obs[GROUP_KEY] == ct]
        mat   = adata[cells].layers['normalized']
        mean_vec = np.asarray(mat.mean(axis=0)).ravel()
        expr[ct] = mean_vec[var_lookup[genes_avail].values]

    # Z-score (每基因行) 并截断到 [0,2]
    expr_z = (
        expr.sub(expr.mean(axis=1), axis=0)
            .div(expr.std(axis=1).replace(0, 1), axis=0)
            .clip(lower=0, upper=2) 
            .T
    ).loc[present_ct]               # (修改) 保证行顺序
    
    row_colors = None
    if f"{GROUP_KEY}_colors" in adata.uns:
        try:
            # 确保 categories 和 colors 列表能匹配
            if pd.api.types.is_categorical_dtype(adata.obs[GROUP_KEY]):
                cat = adata.obs[GROUP_KEY].cat.categories
            else:
                cat = sorted(adata.obs[GROUP_KEY].unique())
                
            palette  = adata.uns[f"{GROUP_KEY}_colors"]
            
            # 如果调色板长度与类别数不匹配，进行适配
            if len(palette) == len(cat):
                lut = dict(zip(cat, palette))
                row_colors = [lut.get(ct, '#808080') for ct in present_ct] # 使用 .get
            else:
                 print(f"  [WARN] 颜色数量 ({len(palette)}) 与类别数 ({len(cat)}) 不匹配。")

        except Exception as e:
            print(f"  [WARN] 提取行颜色失败: {e}")

    # --- 选 Gland 专属色板 (基于基因来源) ---
    cmap_choice = GLAND_CMAP.get(gene_list_tag, HEAT_CMAP)

    sns.set(style="white", font_scale=0.75)
    g = sns.clustermap(
        expr_z,
        row_cluster=False, col_cluster=False,
        cmap=cmap_choice,
        vmin=0, vmax=2,           
        row_colors=row_colors,
        linewidths=0.1,
        figsize=(max(6, len(genes_avail)*0.4), max(3, len(present_ct)*0.45))
    )
    g.ax_heatmap.set_xlabel("Gene")
    g.ax_heatmap.set_ylabel("Cell type")
    g.ax_heatmap.set_title(f"{gene_list_tag} TopGenes on {gland_tag} Data", pad=10)
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90)

    fname = f"{gene_list_tag}_TopGenes_on_{gland_tag}.pdf"
    g.savefig(fname, dpi=DPI, bbox_inches="tight")
    plt.close()
    return fname


# =============================== MAIN (已修改) ===============================
def main():
    # (修改) 1. 加载、拆分并获取细胞类型列表
    ads_dict, all_celltypes = load_and_split_anndata()
    if not ads_dict:
        print("[ERROR] 未能加载或拆分 AnnData。退出。")
        return
        
    top_dict = {}

    # (修改) 2. 按 Gland 筛选基因
    for gland in GLANDS:
        if gland not in ads_dict:
            continue
        print(f"\n[ {gland} ] 筛选腺体特异基因 …")
        
        # (修改) 传入 all_celltypes
        ct_deg = filter_gland_specific(gland, ads_dict, all_celltypes) 
        csvfile = save_deg_csv(ct_deg, gland)
        if csvfile:
            print(f"  CSV → {csvfile}")
            
        # (修改) 传入 all_celltypes
        top_dict[gland] = top_genes(ct_deg, all_celltypes) 
        print(f"  Top genes ({len(top_dict[gland])}) collected.")

    # (修改) 3. 绘图 (N x N 张)
    print("\n[ PLOTTING ] 开始绘制所有热图...")
    plot_count = 0
    for gene_gland, genes in top_dict.items(): 
        if not genes:
            print(f"  [INFO] {gene_gland} 基因列表为空，跳过绘图。")
            continue
        
        for data_gland, ad in ads_dict.items(): 
            
            # (修改) 传入 all_celltypes
            fname = plot_heatmap(genes, ad, data_gland, gene_gland, all_celltypes)
            if fname:
                print(f"  Heatmap saved: {fname}")
                plot_count += 1
            else:
                print(f"  Heatmap skipped: {gene_gland}_genes on {data_gland}_data")

    print(f"\n✅ 任务完成！共生成 {len(top_dict)} 个 CSV、{plot_count} 张 heatmap。")

if __name__ == "__main__":
    main()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18388\2140682397.py:52: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(ad.obs["newcelltype"]):



[ M ] 筛选物种特异基因 …
  CSV → M_specific_DEG.csv
  Top genes (25) collected.

[ R ] 筛选物种特异基因 …
  CSV → R_specific_DEG.csv
  Top genes (25) collected.

[ S ] 筛选物种特异基因 …
  CSV → S_specific_DEG.csv
  Top genes (25) collected.
  Heatmap saved: M_TopGenes_on_M-MG.pdf
  Heatmap saved: M_TopGenes_on_R-MG.pdf
  Heatmap saved: M_TopGenes_on_S-MG.pdf
  Heatmap saved: R_TopGenes_on_M-MG.pdf
  Heatmap saved: R_TopGenes_on_R-MG.pdf
  Heatmap saved: R_TopGenes_on_S-MG.pdf
  Heatmap saved: S_TopGenes_on_M-MG.pdf
  Heatmap saved: S_TopGenes_on_R-MG.pdf
  Heatmap saved: S_TopGenes_on_S-MG.pdf

✅ 任务完成！共生成 3 个 CSV、9 张 heatmap。


In [7]:
# ---------------------------------------------------------------
# Imports & constants
# ---------------------------------------------------------------
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

data_dir = Path(".")          # ← 若 CSV 不在当前目录，请改这里

CELLTYPE_ORDER = ["LumSEC-Lac", "LumSEC-Lip", "LumHR", "Basal", "MaSC"]
DATASETS       = {"M": "M-MG", "R": "R-MG", "S": "S-MG"}

label_colors = {
    "Basal":        "#1f78b4",
    "LumHR":        "#ff7f00",
    "LumSEC-Lac":   "#6a3d9a",
    "LumSEC-Lip":   "#b15928",
    "MaSC":         "#33a02c",
}
hatches = {"M": "", "R": "//", "S": "++"}

PDF_DPI        = 300
CONS_BAR_H     = 0.6                 # 厚度（右图单条）
SPEC_BAR_H     = CONS_BAR_H / 3      # 左图单条 = 右图总厚度 / 3
# ---------------------------------------------------------------
# 1) 统计物种特异 (logFC>0)
# ---------------------------------------------------------------
spec_counts = {}
for sp in DATASETS:
    f = data_dir / f"{sp}_specific_DEG.csv"
    df = pd.read_csv(f)
    df = df[df["logfoldchange"] > 0]
    counts = (df.groupby("celltype")
                .size()
                .reindex(CELLTYPE_ORDER)
                .fillna(0)
                .astype(int))
    spec_counts[sp] = counts
spec_df = pd.DataFrame(spec_counts)
# ---------------------------------------------------------------
# 2) 统计保守基因
# ---------------------------------------------------------------
cons_f = data_dir / "conserved_gene_expression.csv"
cons_df = pd.read_csv(cons_f)
cons_counts = (cons_df.groupby("celltype")
                         .size()
                         .reindex(CELLTYPE_ORDER)
                         .fillna(0)
                         .astype(int))
# ---------------------------------------------------------------
# 3) 绘图
# ---------------------------------------------------------------
y_pos = range(len(CELLTYPE_ORDER))
fig, (axL, axR) = plt.subplots(
    1, 2, figsize=(12, 4.5),
    sharey=True,
    gridspec_kw={"width_ratios": [2, 1]}
)

# ---- 左：物种特异（镜像、合适厚度） ----
for i, sp in enumerate(DATASETS):
    offset = (i - 1) * SPEC_BAR_H          # -SPEC, 0, +SPEC
    axL.barh(
        [y + offset for y in y_pos],
        -spec_df[sp],                      # 负值 → 向左
        height=SPEC_BAR_H,
        color=[label_colors[ct] for ct in CELLTYPE_ORDER],
        hatch=hatches[sp],
        edgecolor="black", linewidth=0.6
    )

# 去掉左图 y 轴标签和左侧 spine
axL.set_yticks([])
axL.set_ylabel("")
axL.spines["left"].set_visible(False)

axL.set_xlabel("number of markers")
axL.set_title("Species-specific (logFC > 0)")
axL.invert_yaxis()

# 将负刻度标签显示为正数
axL.set_xticklabels([abs(int(x)) for x in axL.get_xticks()])

# 图例：空心 patch
legend_handles = [
    mpatches.Patch(facecolor="none",
                   hatch=hatches[sp],
                   edgecolor="black",
                   label=f"{sp} ({DATASETS[sp]})")
    for sp in DATASETS
]
axL.legend(handles=legend_handles,
           frameon=False, loc="upper left")

# ---- 右：保守 ----
axR.barh(
    y_pos,
    cons_counts,
    height=CONS_BAR_H,
    color=[label_colors[ct] for ct in CELLTYPE_ORDER],
    edgecolor="black", linewidth=0.6
)
axR.set_xlabel("number of markers")
axR.set_title("Conserved")
axR.invert_yaxis()

# ---- 坐标轴 & 边框处理 ----
for ax in (axL, axR):
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.8)

# 右图保留 ytick 标签
axR.set_yticks(y_pos)
axR.set_yticklabels(CELLTYPE_ORDER)

plt.tight_layout()
plt.savefig("marker_counts_combined.pdf", dpi=PDF_DPI)
plt.close()

print("✓ marker_counts_combined.pdf 已生成")


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18388\389711897.py:83: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axL.set_xticklabels([abs(int(x)) for x in axL.get_xticks()])


✓ marker_counts_combined.pdf 已生成
